# Workshop: Gemma from Scratch
## Notebook 7: The Transformer Block

**Estimated Time: 20 minutes**

Now we assemble the pieces! A Transformer block (or layer) is the fundamental repeating unit of the model. It consists of an **Attention sub-block** and an **MLP sub-block**, connected by residual connections.

### Learning Objectives:
1. Assemble Attention, MLP, and RMSNorm into a single layer.
2. Implement residual (skip) connections.
3. Understand the data flow through the block.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

# Redefine necessary components to make this notebook self-contained
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return (x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight

class SimpleGQA(nn.Module):
    def __init__(self, d_in, n_heads, n_kv_groups, h_dim):
        super().__init__()
        self.n_heads, self.n_kv_groups, self.h_dim = n_heads, n_kv_groups, h_dim
        self.group_size = n_heads // n_kv_groups
        self.W_q = nn.Linear(d_in, n_heads * h_dim, bias=False)
        self.W_k = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.W_v = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * h_dim, d_in, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        q = self.W_q(x).view(B, T, self.n_heads, self.h_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.h_dim)
        weights = torch.softmax(scores, dim=-1)
        out = torch.matmul(weights, v)
        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.h_dim)
        return self.out_proj(out)

class GatedMLP(nn.Module):
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.gate_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.up_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.down_proj = nn.Linear(d_hidden, d_in, bias=False)
    def forward(self, x):
        return self.down_proj(F.gelu(self.gate_proj(x), approximate="tanh") * self.up_proj(x))

### 1. The Block Architecture

The data flow is:
1. `residual = x` (Store input)
2. `x = RMSNorm(x)` (Pre-norm)
3. `x = Attention(x)`
4. `x = RMSNorm(x)` (Post-norm - unique to this Gemma implementation)
5. `x = x + residual` (Add residual)

And then the same pattern for the MLP.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, n_heads, n_kv_groups, h_dim, hidden_dim):
        super().__init__()
        # Sub-layers
        self.attn = SimpleGQA(dim, n_heads, n_kv_groups, h_dim)
        self.ffn = GatedMLP(dim, hidden_dim)
        
        # Norm layers
        self.input_layernorm = RMSNorm(dim)
        self.post_attention_layernorm = RMSNorm(dim)
        self.pre_feedforward_layernorm = RMSNorm(dim)
        self.post_feedforward_layernorm = RMSNorm(dim)

    def forward(self, x):
        # --- Attention Sub-block ---
        shortcut = x
        x = self.input_layernorm(x)
        x = self.attn(x)
        x = self.post_attention_layernorm(x)
        x = shortcut + x
        
        # --- MLP Sub-block ---
        shortcut = x
        x = self.pre_feedforward_layernorm(x)
        x = self.ffn(x)
        x = self.post_feedforward_layernorm(x)
        x = shortcut + x
        
        return x

dim, n_h, n_kv, h_d, h_ff = 128, 8, 2, 16, 512
block = TransformerBlock(dim, n_h, n_kv, h_d, h_ff)
x = torch.randn(1, 10, dim)
out = block(x)
print("Output shape:", out.shape)
assert out.shape == x.shape

### 2. The Power of Residual Connections

Residual connections ($x + f(x)$) allow gradients to flow directly through the network, bypassing the non-linearities of the attention and MLP layers. This is what makes it possible to train models with hundreds of layers.

### Exercise:
What would happen if we didn't have the `RMSNorm` layers? Why is normalization so important before (and after) each sub-layer?

<details>
<summary><b>Click to see answer</b></summary>

Without normalization, the activations would grow uncontrollably in magnitude as they pass through multiple layers. This leads to "exploding gradients" and makes training unstable. `RMSNorm` keeps the activations at a consistent scale (roughly unit variance), which ensures smoother gradient flow and faster convergence.
</details>